In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('iceberg-test') \
    .getOrCreate()

SLF4J: Class path contains multiple SLF4J providers.
SLF4J: Found provider [org.apache.logging.slf4j.SLF4JServiceProvider@11a9e7c8]
SLF4J: Found provider [org.slf4j.impl.JBossSlf4jServiceProvider@3901d134]
SLF4J: See https://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual provider is of type [org.apache.logging.slf4j.SLF4JServiceProvider@11a9e7c8]


08:56:24.459 [main] ERROR javax.management.mbeanserver - The LogManager accessed before the "java.util.logging.manager" system property was set to "org.jboss.logmanager.LogManager". Results may be unexpected.


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/01 08:56:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.sparkContext.getConf().getAll()

[('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog'),
 ('spark.sql.defaultCatalog', 'nessie'),
 ('spark.app.submitTime', '1730451385263'),
 ('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false'),
 ('spark.sql.catalog.nessie.uri', 'http://nessie:19120/api/v1'),
 ('spark.hadoop.fs.s3a.path.style.access', 'true'),
 ('spark.hadoop.fs.s3a.secret.key', 'password'),
 ('spark.sql.catalog.nessie.ref', 'main'),
 ('spark.driver.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch

In [5]:
s3_path = 's3a://nyc-project/raw_data/'

In [6]:
df = spark.read.parquet(f'{s3_path}/2019/01/*')

24/11/01 08:56:40 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [7]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

In [10]:
spark.catalog.createNamespace("default")

AttributeError: 'Catalog' object has no attribute 'createNamespace'

In [11]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS default")

DataFrame[]

In [12]:
df.write.format('iceberg').mode('overwrite').saveAsTable('default.my_iceberg_table')

In [14]:
df_iceberg = spark.read.format('iceberg').load('default.my_iceberg_table')

In [15]:
df_iceberg.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6